# 06 — Model training and cross-validation
**Business objective:** Estimate used-car selling prices in INR from six listing attributes. We will select a candidate based on grouped cross-validation, **not the held-out test set**. Run notebooks from the `notebooks/` directory.

**Data decision:** Duplicate feature combinations may represent different cars; we retain rows but keep identical feature combinations in the same group during splitting. `name` is excluded from this first baseline.

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "processed" / "cleaned_cars.csv"
if not DATA.exists():
    DATA = ROOT / "data" / "raw" / "car_details.csv"
ARTIFACTS = ROOT / "models"
ARTIFACTS.mkdir(exist_ok=True)
REPORTS = ROOT / "reports"
REPORTS.mkdir(exist_ok=True)
df = pd.read_csv(DATA)
print("Dataset:", DATA.name, "| rows:", len(df))

Dataset: car_details.csv | rows: 4340


In [2]:
features = ["year", "km_driven", "fuel", "seller_type", "transmission", "owner"]
target = "selling_price"
assert set(features + [target]).issubset(df.columns), "Missing expected columns"
assert df[features + [target]].notna().all().all(), "Resolve missing data before training"
assert (df[target] > 0).all(), "Selling price must be positive"
X = df[features].copy()
y = df[target].copy()
# Identical input combinations stay together, even if their prices differ.
groups = pd.util.hash_pandas_object(X, index=False)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]
assert set(groups_train).isdisjoint(set(groups.iloc[test_idx]))
print(f"Train: {len(X_train):,} | Holdout: {len(X_test):,}")
# Persist indices so every notebook evaluates the exact same split.
joblib.dump({"train_idx": train_idx, "test_idx": test_idx}, ARTIFACTS / "split_indices.joblib")

Train: 3,448 | Holdout: 892


['/mnt/data/used_car_price_prediction/models/split_indices.joblib']

## Why these models?
- **Dummy:** establishes the error of predicting the typical price.
- **Ridge:** linear, regularized benchmark.
- **Random forest:** nonlinear interactions without heavy tuning.
- **Gradient boosting:** another strong tabular baseline.

We scale numeric values for Ridge and one-hot encode categories for all models. All transformations are fitted **inside each training fold**.

In [3]:
numeric_features = ["year", "km_driven"]
categorical_features = ["fuel", "seller_type", "transmission", "owner"]
def make_preprocessor():
    return ColumnTransformer([
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
    ])

def make_pipeline(estimator):
    return Pipeline([("preprocess", make_preprocessor()), ("regressor", estimator)])

candidates = {
    "Dummy": DummyRegressor(strategy="median"),
    "Ridge": Ridge(alpha=10),
    "RandomForest": RandomForestRegressor(n_estimators=200, min_samples_leaf=2, random_state=42, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingRegressor(max_iter=200, random_state=42)
}
cv = GroupKFold(n_splits=5)

In [4]:
# Cell 6 — Train and evaluate with TRAINING-ONLY cross-validation.
# We do not touch X_test/y_test until the final evaluation notebook.
rows = []
for model_name, estimator in candidates.items():
    pipeline = make_pipeline(estimator)
    scores = cross_validate(
        pipeline, X_train, y_train, groups=groups_train, cv=cv,
        scoring={"mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error", "r2": "r2"},
        n_jobs=1, error_score="raise"
    )
    rows.append({
        "model": model_name,
        "cv_mae_mean": -scores["test_mae"].mean(),
        "cv_mae_std": scores["test_mae"].std(),
        "cv_rmse_mean": -scores["test_rmse"].mean(),
        "cv_r2_mean": scores["test_r2"].mean()
    })
results = pd.DataFrame(rows).sort_values("cv_mae_mean").reset_index(drop=True)
print(results.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))
results.to_csv(REPORTS / "cv_model_comparison.csv", index=False)

               model  cv_mae_mean  cv_mae_std  cv_rmse_mean  cv_r2_mean
        RandomForest   203,644.25   24,638.77    416,246.19        0.46
HistGradientBoosting   205,242.59   22,687.48    413,191.28        0.46
               Ridge   233,102.28   25,881.65    425,687.37        0.44
               Dummy   300,715.62   35,450.49    592,331.70       -0.07


In [5]:
winner_name = results.loc[0, "model"]
print("Lowest CV MAE:", winner_name)
print("Difference vs Dummy (INR):", round(
    results.set_index("model").loc["Dummy", "cv_mae_mean"] - results.loc[0, "cv_mae_mean"], 2
))
# Fit candidate on all training data; still not evaluated on holdout.
winning_pipeline = make_pipeline(candidates[winner_name])
winning_pipeline.fit(X_train, y_train)
joblib.dump(winning_pipeline, ARTIFACTS / "baseline_candidate.joblib")
(REPORTS / "selected_model.txt").write_text(winner_name, encoding="utf-8")
print("Saved candidate pipeline. Next: 07_model_evaluation.ipynb")

Lowest CV MAE: RandomForest
Difference vs Dummy (INR): 97071.37


Saved candidate pipeline. Next: 07_model_evaluation.ipynb


### Analyst notes to complete after running
1. Which model has the lowest grouped-CV MAE? What is the error in rupees?
2. Is it meaningfully better than the Dummy baseline?
3. How variable is MAE across folds (`cv_mae_std`)?
4. Do **not** report cross-validation results as test-set performance.